### 1. Подготовка датасета

Целью тренировки нейросети является создание переводчика с нейтрального русского языка на более токсичный вариант.<br>

За основу был взят датасет [Toxic Russian Comments](https://www.kaggle.com/datasets/alexandersemiletov/toxic-russian-comments) c Kaggle. Он содержит комментарии оставленные пользователями в соцсети **ok.ru**, они помечены такими метками как:<br>
 - NORMAL - содержит нейтральные комментарии<br>
 - INSULT - содержит уничижительные комментарии<br>
 - THREAT - содержит комментарии с угрозами<br>
 - OBSCENITY - содержит комментарии с угрозами сексуального характера<br>

Для формирования датасета были выбраны все комментарии кроме нейтральных, длинной не более 30 слов (знаки препинания учитывались как слова). Далее каждый комментарий прошел через инференс LLM [GigaChat3.1-10B-A1.8B](https://huggingface.co/ai-sage/GigaChat3.1-10B-A1.8B-GGUF) с системным промптом указывающим перевести комментарий в более нейтральную форму. В итоге получился сырой датасет в форме<br>
<br>токсичный комментарий **\t** нейтральный комментарий<br><br>
   Стоит отметить, что использование небольшой зацензуренной LLM для автоматической разметки привело к существенному зашумлению итогового датасета вследствии следующих проблем - галлюцинаций нейросети, цензуры нейросети, неоптимального системного промпта, неоптимального выбора самой нейросети для инференса.<br>
Как следствие данный ноутбук иллюстрирует принцип GARBAGE IN - GARBAGE OUT.

In [2]:
from io import open
import unicodedata
import string
import re
import random
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import warnings

Очистим фразы от лишних символов, оставляя только буквы и знаки препинания и сформируем пары нейтральный/токсичный комментарий:

In [3]:
import re

def clean_text(text):
    text = text.lower().strip()
    # Изолируем знаки препинания пробелами (чтобы ", " стало отдельным токеном)
    text = re.sub(r"([,.!?])", r" \1 ", text)
    # Оставляем только кириллицу и знаки препинания
    text = re.sub(r"[^а-яА-ЯёЁ,.!?]+", r" ", text)
    # Убираем лишние пробелы
    return " ".join(text.split()).strip()

pairs = []
with open('toxic_dataset.tsv', 'r', encoding='utf-8') as f:
    for line in f:
        if '\t' in line:
            parts = line.split('\t')
            if len(parts) == 2:
                # порядок фраз в датасете "токсичная/нейтральная" поэтому сначала берем вторую часть строки
                pairs.append([clean_text(parts[1]), clean_text(parts[0])])

print(f"Загружено пар: {len(pairs)}")

Загружено пар: 40822


На данный момент общий словарь датасета вмещает около 73000 токенов, что невероятно много. Отдельно для нейтральных и токсичных комментариев выбираем слова которые встречаются 2 и менее раз и с помощью векторного представления всех слов с помощью BERT модели [rubert-tiny2](https://huggingface.co/cointegrated/rubert-tiny2) им находятся аналоги, что поможет значительно уменьшить размер словарей.

Разделям словари на часто встречаемые и редко встречаемые слова:

In [4]:
input_counts = {}
target_counts = {}

for pair in pairs:
    # Считаем слова для входа
    for word in pair[0].split():
        input_counts[word] = input_counts.get(word, 0) + 1
    # Считаем слова для выхода
    for word in pair[1].split():
        target_counts[word] = target_counts.get(word, 0) + 1

min_count = 3

# Разделяем слова на частые (>=3) и редкие (<3)
input_freq = [w for w, c in input_counts.items() if c >= min_count]
input_rare = [w for w, c in input_counts.items() if c < min_count]

target_freq = [w for w, c in target_counts.items() if c >= min_count]
target_rare = [w for w, c in target_counts.items() if c < min_count]

print(f"Вход (Нейтральный): частых {len(input_freq)}, редких {len(input_rare)}")
print(f"Выход (Токсичный): частых {len(target_freq)}, редких {len(target_rare)}")

Вход (Нейтральный): частых 13119, редких 30679
Выход (Токсичный): частых 12801, редких 44555


### Строим словари замен с учетом косинусной близости векторов:

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('cointegrated/rubert-tiny2').cuda()

def build_semantic_correction_map(freq_words, rare_words, threshold=0.75):
    freq_embeddings = model.encode(freq_words, convert_to_tensor=True, batch_size=2048)
    rare_embeddings = model.encode(rare_words, convert_to_tensor=True, batch_size=2048)
    
    # Считаем косинусное расстояние
    cosine_scores = util.cos_sim(rare_embeddings, freq_embeddings)
    best_scores, best_indices = torch.max(cosine_scores, dim=1)
    
    c_map = {}
    for i, rare in enumerate(rare_words):
        if best_scores[i].item() >= threshold:
            c_map[rare] = freq_words[best_indices[i]]
        else:
            c_map[rare] = "UNK"
    return c_map

print("Строим карту замен для входа...")
input_map = build_semantic_correction_map(input_freq, input_rare, threshold=0.75)

print("Строим карту замен для выхода...")
target_map = build_semantic_correction_map(target_freq, target_rare, threshold=0.75)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Строим карту замен для входа...
Строим карту замен для выхода...


### Применяем полученные словари соотвественно фразам из датасета:

In [6]:
def apply_corrections(sentence, freq_words_list, correction_map):
    freq_set = set(freq_words_list)
    new_words = []
    
    for word in sentence.split():
        if word in freq_set:
            new_words.append(word)
        else:
            new_words.append(correction_map.get(word, "UNK"))
            
    return " ".join(new_words)

final_pairs = []
for pair in pairs:
    clean_in = apply_corrections(pair[0], input_freq, input_map)
    clean_out = apply_corrections(pair[1], target_freq, target_map)
    final_pairs.append([clean_in, clean_out])

print("Датасет очищен и нормализован")

Датасет очищен и нормализован


### Создаем класс словаря:

In [7]:
# Определим по умолчанию 2 токена которые будут нам информировать о начале предложения и конце предложения (SOS и EOS):
SOS_token = 0
EOS_token = 1

class LanguageVocabulary:
    def __init__(self, name):
        self.name = name
        self.word2index = {"SOS": 0, "EOS": 1, "UNK": 2}
        self.word2count = {"SOS": 0, "EOS": 0, "UNK": 0}
        self.index2word = {0: "SOS", 1: "EOS", 2: "UNK"}
        self.n_words = 3

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

### Заполняем словари нормализованными данными

In [8]:
input_lang = LanguageVocabulary('neutral')
output_lang = LanguageVocabulary('toxic')

for pair in final_pairs:
    input_lang.add_sentence(pair[0])
    output_lang.add_sentence(pair[1])

print(f"Размер входного словаря: {input_lang.n_words}")
print(f"Размер выходного словаря: {output_lang.n_words}")

Размер входного словаря: 13122
Размер выходного словаря: 12804


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Создаем функции для кодирования для передачи в модель:

In [10]:
def tensorFromSentence(lang, sentence):
    # Если слова нет, ставим 2 (UNK)
    indexes = [lang.word2index.get(word, 2) for word in sentence.split(' ')]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

### Создаем классы для энкодера и декодера:

In [11]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=3, dropout=0.2):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(input_size, hidden_size)
        # Добавляем глубину и дропаут
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers=self.num_layers, dropout=dropout)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(self.num_layers, 1, self.hidden_size, device=device)

In [12]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, num_layers=3, dropout=0.2):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers=self.num_layers, dropout=dropout)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = nn.functional.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

### Создадим функцию обучения для работы только с ОДНОЙ парой:

In [13]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=20):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < 0.5 else False

    if use_teacher_forcing:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

### Функция которая будет пробегаться по всем парам и использовать эти пары для обучения сети:

In [14]:
def trainIters(encoder, decoder, n_iters, print_every=1000, learning_rate=0.001):
    start = time.time()
    plot_losses = []
    print_loss_total = 0 

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    
    # Готовим случайные пары для обучения
    training_pairs = [tensorsFromPair(random.choice(final_pairs)) for i in range(n_iters)]
    criterion = nn.NLLLoss() # Так как в декодере LogSoftmax

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print(f'Итерация: {iter} | Loss: {print_loss_avg:.4f}')

### Функция для инференса:

In [15]:
def evaluate(encoder, decoder, sentence, max_length=30):
    with torch.no_grad(): # Отключаем градиенты для скорости
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        # Кодируем вход
        for ei in range(input_length):
            _, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

        # Стартуем декодер с SOS_token
        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        
        decoded_words = []
        for di in range(max_length):
            output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = output.data.topk(1)
            
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
            
            decoder_input = topi.squeeze().detach()
            
        return ' '.join(decoded_words)

### Этап обучения:

In [81]:
hidden_size = 1024
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

EncoderRNN(
  (embedding): Embedding(13122, 1024)
  (gru): GRU(1024, 1024, num_layers=3, dropout=0.2)
)


In [172]:
trainIters(encoder, decoder, n_iters=200000, print_every=20000, learning_rate=0.00005)

Итерация: 20000 | Loss: 2.5244
Итерация: 40000 | Loss: 2.4733
Итерация: 60000 | Loss: 2.3983
Итерация: 80000 | Loss: 2.3324
Итерация: 100000 | Loss: 2.2856
Итерация: 120000 | Loss: 2.2496
Итерация: 140000 | Loss: 2.1952
Итерация: 160000 | Loss: 2.1625
Итерация: 180000 | Loss: 2.1185
Итерация: 200000 | Loss: 2.0674


Сохраняем полученные веса:

In [224]:
torch.save(encoder.state_dict(), 'toxic_encoder_perfect.pth')
torch.save(decoder.state_dict(), 'toxic_decoder_perfect.pth')

### Инференс:

In [21]:
encoder.eval()
decoder.eval()
output_sentence = evaluate(encoder, decoder, 'болтливый человек')
print(output_sentence)

ебанутый долбаёб <EOS>


### Код для загрузки предобученной модели:

In [19]:
import os
import torch

encoder_path = 'toxic_encoder_perfect.pth'
decoder_path = 'toxic_decoder_perfect.pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if os.path.exists(encoder_path) and os.path.exists(decoder_path):

    hidden_size = 1024
    encoder = EncoderRNN(input_lang.n_words, hidden_size, num_layers=3).to(device)
    decoder = DecoderRNN(hidden_size, output_lang.n_words, num_layers=3).to(device)
    
    encoder.load_state_dict(torch.load(encoder_path, map_location=device))
    decoder.load_state_dict(torch.load(decoder_path, map_location=device))
    
    encoder.eval()
    decoder.eval()

### Проверка качества с помощью метрики BLEU
не применялась